# 🦜 Colombian Bird Song — Audio Pre-Processing Pipeline

This notebook prepares the audio dataset for model training. It fetches bird recording metadata from [Xeno-Canto](https://xeno-canto.org), filters recordings and species, downloads MP3 files, and produces normalised log-Mel spectrogram tensors ready for training.

---

## Pipeline Overview

| Step | Description | Status |
|------|-------------|--------|
| 0 | Setup & Authentication | ✅ |
| 1 | Metadata download & filtering | ✅ |
| 2 | Audio download (asyncio) | ✅ |
| 3 | Audio cleaning — VAD + DSP | ✅ |
| 4 | RMS normalisation | ✅ |
| 5 | Chunking (3 s, 50 % overlap) | ✅ |
| 6 | Log-Mel spectrograms → normalised tensors | ✅ |

---

## Step 0 — Setup & Authentication

Install required packages (run once) and configure your Xeno-Canto API key.

Generate the key from your account page: https://xeno-canto.org/account/api

In [ ]:
# Run once — uncomment if packages are missing
#!pip install pyarrow tqdm aiohttp imageio_ffmpeg nest_asyncio soundfile scipy -q

In [ ]:
import os

XC_API_KEY = os.environ.get("XC_API_KEY", "")
if not XC_API_KEY:
    XC_API_KEY = input("Enter your Xeno-Canto API key: ").strip()

if not XC_API_KEY:
    raise ValueError("XC_API_KEY is not set. Configure it to continue.")

In [ ]:
# 1.1  Deep-learning backend — automatic detection (PyTorch w/ CUDA → TensorFlow)

try:
    import torch as tr
    if tr.cuda.is_available():
        DL_BACKEND = "pytorch"
        DEVICE     = "cuda"
        print(f"PyTorch {tr.__version__}  |  CUDA: {tr.cuda.is_available()}  |  GPU: {tr.cuda.get_device_name(0)}")
    else:
        raise ImportError("CUDA not available for PyTorch.")
except ImportError:
    try:
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers
        DL_BACKEND = "tensorflow"
        print(f"TensorFlow {tf.__version__}  |  Keras {keras.__version__}  |  GPU: {tf.config.list_physical_devices('GPU')}")
    except ImportError:
        DL_BACKEND = None
        print("Warning: neither PyTorch+CUDA nor TensorFlow found. Deep-learning features unavailable.")

## ⚙️ Environment Configuration

Set `ONLINE = True` to run on Colab with Google Drive storage, `False` for local use.

Set `CLEAN_RUN = True` to wipe all previous output from Drive before starting.

In [ ]:
from pathlib import Path
import shutil

# ── Environment ──────────────────────────────────────────────────
ONLINE = False
if ONLINE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/bird_project")
else:
    PROJECT_DIR = Path.cwd() / "bird_project"
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ── Directory structure ──────────────────────────────────────────
DATA_DIR      = PROJECT_DIR / "data"
RAW_DIR       = DATA_DIR   / "raw"
PROCESSED_DIR = DATA_DIR   / "processed"
AUDIO_DIR     = PROJECT_DIR / "bird_songs"
MODEL_DIR     = PROJECT_DIR / "models"

# ── Xeno-Canto paths ─────────────────────────────────────────────
RAW_PARQUET_PATH      = DATA_DIR / "colombia_birds_raw.parquet"
FILTERED_PARQUET_PATH = DATA_DIR / "colombia_birds_filtered.parquet"

# ── Run flags ────────────────────────────────────────────────────
FORCE_REFRESH = False
MAX_WORKERS   = 8

# ── Clean run (wipes Drive output) ───────────────────────────────
# Folders removed when CLEAN_RUN = True:
#   bird_songs/   — downloaded MP3s
#   processed/    — all DSP output (clean, normalised, chunks, spectrograms)
#   models/       — saved model artefacts
CLEAN_RUN = False

CLEAN_TARGETS = [
    (AUDIO_DIR,     "bird_songs (MP3s)"),
    (PROCESSED_DIR, "processed (all DSP output)"),
    (MODEL_DIR,     "models"),
]

if CLEAN_RUN:
    for target_path, label in CLEAN_TARGETS:
        if target_path.exists():
            shutil.rmtree(target_path)
            target_path.mkdir(parents=True, exist_ok=True)
            print(f"Cleared : {label}")
        else:
            print(f"Not found (nothing to clear): {label}")
    print("Clean run ready.")
else:
    found = [label for path, label in CLEAN_TARGETS if path.exists()]
    if found:
        print("Resuming with existing data:")
        for label in found:
            print(f"  ✓ {label}")
    else:
        print("No existing data found — starting fresh.")

# Create all directories
for directory in [DATA_DIR, RAW_DIR, PROCESSED_DIR, AUDIO_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"\nPROJECT_DIR   : {PROJECT_DIR}")
print(f"DATA_DIR      : {DATA_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"AUDIO_DIR     : {AUDIO_DIR}")
print(f"MAX_WORKERS   : {MAX_WORKERS}")
print(f"FORCE_REFRESH : {FORCE_REFRESH}")

## Block 1 — Metadata Download & Filtering

Fetches all Colombian bird recordings from the Xeno-Canto API v3 (`cnt:colombia grp:birds`), validates country / group / bounding-box coordinates, and filters by quality (A/B), duration (3–60 s), and minimum recording count per species (≥ 35).

In [ ]:
# =============================================================
# BLOCK 1: Metadata download & filtering
# =============================================================

import logging
import requests
import pandas as pd
from tqdm import tqdm

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

BASE_URL = "https://xeno-canto.org/api/3/recordings"
QUERY    = "cnt:colombia grp:birds"
PER_PAGE = 500

COL_LAT_MIN, COL_LAT_MAX = -4.5,  13.0
COL_LNG_MIN, COL_LNG_MAX = -82.0, -66.5

KEEP_COLUMNS = [
    "id", "gen", "sp", "ssp", "en",
    "cnt", "loc", "lat", "lon",
    "q", "length", "date", "file", "type"
]

# ── HTTP client ──────────────────────────────────────────────

def _make_session(api_key: str = "") -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "Accept":     "application/json",
        "User-Agent": "BirdDatasetBuilder/1.0",
    })
    s._xc_api_key = api_key
    return s

SESSION = _make_session(XC_API_KEY)

# ── Pagination ────────────────────────────────────────────────

def _fetch_page(page: int) -> dict:
    params = {
        "query":    QUERY,
        "page":     page,
        "per_page": PER_PAGE,
    }
    if SESSION._xc_api_key:
        params["key"] = SESSION._xc_api_key
    try:
        r = SESSION.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:
        log.error(f"Error on page {page}: {e}")
        return {"recordings": []}

def fetch_all_recordings() -> list[dict]:
    first     = _fetch_page(1)
    num_pages = int(first.get("numPages", 1))
    log.info(
        f"Total: {first.get('numRecordings','?')} recordings, "
        f"{first.get('numSpecies','?')} spp, {num_pages} pages"
    )
    page_cache = {1: first}
    all_recs   = []
    for page in tqdm(range(1, num_pages + 1), desc="Downloading metadata"):
        data = page_cache.pop(page, None) or _fetch_page(page)
        all_recs.extend(data.get("recordings", []))
    return all_recs

# ── Filters ───────────────────────────────────────────────────

def _in_colombia_bbox(lat, lon) -> bool:
    try:
        return (
            COL_LAT_MIN <= float(lat) <= COL_LAT_MAX
            and COL_LNG_MIN <= float(lon) <= COL_LNG_MAX
        )
    except (TypeError, ValueError):
        return False  # reject if coordinates are missing or invalid

def validate_and_filter(df: pd.DataFrame) -> pd.DataFrame:
    mask = (
        df["cnt"].fillna("").str.lower().eq("colombia")
        & df["grp"].fillna("").str.lower().eq("birds")
    )
    bbox_mask = pd.Series(
        [_in_colombia_bbox(lat, lon) for lat, lon in zip(df["lat"], df["lon"])],
        index=df.index
    )
    quality_mask  = df["q"].isin(["A", "B"])
    df["duration_sec"] = (
        pd.to_timedelta("00:" + df["length"].fillna("00:00"))
        .dt.total_seconds()
    )
    duration_mask = df["duration_sec"].between(3, 60)
    url_mask      = df["file"].notna()

    df = df[mask & bbox_mask & quality_mask & duration_mask & url_mask].copy()

    df["scientific_name"] = (
        df["gen"].fillna("") + " " + df["sp"].fillna("")
    ).str.strip()

    counts = df["scientific_name"].value_counts()
    df = df[df["scientific_name"].isin(counts[counts >= 35].index)].copy()

    available = [c for c in KEEP_COLUMNS if c in df.columns]
    return df[available + ["duration_sec", "scientific_name"]]

# ── Execution ─────────────────────────────────────────────────

log.info("=" * 60)
log.info("BLOCK 1 — Metadata")
log.info("=" * 60)

if RAW_PARQUET_PATH.exists() and not FORCE_REFRESH:
    log.info("Metadata already cached — loading from disk...")
    df_raw = pd.read_parquet(RAW_PARQUET_PATH)
else:
    log.info("Downloading metadata desde Xeno-Canto...")
    all_recs = fetch_all_recordings()
    df_raw   = pd.DataFrame(all_recs)
    for col in df_raw.select_dtypes("object").columns:
        df_raw[col] = df_raw[col].map(
            lambda v: str(v) if isinstance(v, (dict, list, set, tuple)) else v
        )
    df_raw.to_parquet(RAW_PARQUET_PATH, index=False, engine="pyarrow")
    log.info(f"Raw data saved to: {RAW_PARQUET_PATH}")

df_filtered = validate_and_filter(df_raw)
df_filtered.to_parquet(FILTERED_PARQUET_PATH, index=False)

log.info(f"Filtered records : {len(df_filtered):,}")
log.info(f"Valid species    : {df_filtered['scientific_name'].nunique():,}")
log.info(f"Filtered data saved to: {FILTERED_PARQUET_PATH}")

In [ ]:
df = pd.read_parquet(FILTERED_PARQUET_PATH)
print(f"Before : {df.shape}")

# ── Remove unlabelled species ────────────────────────────
df = df[df["scientific_name"] != "Mystery mystery"].copy()
print(f"After  : {df.shape}")

# ── Verification ──────────────────────────────────────────────
df_obj = df.select_dtypes(include=["object", "string", "category"])
display(df_obj.describe())
print("Species:", df["scientific_name"].nunique())
display(df["scientific_name"].value_counts().sort_values(ascending=True))

# ── Save cleaned parquet ────────────────────────────────────
df.to_parquet(FILTERED_PARQUET_PATH, index=False)
print(f"Parquet updated at: {FILTERED_PARQUET_PATH}")



## Step 1.5 — Bird Cards (Ornithological Metadata)

Builds a resumable JSON catalogue (`bird_card/colombian_birds.json`) with
description, image, conservation status, and taxonomy for every species in
the filtered dataset. Species already present in the file are skipped on
re-runs. Sources are queried in priority order: Wikipedia ES → iNaturalist
→ GBIF → Wikipedia EN. Conservation status is resolved through iNaturalist,
Wikipedia taxobox, Wikidata, and a SPARQL fallback.

In [ ]:
# =============================================================
# STEP 1.5 — Bird Cards 
# =============================================================

import json
import re
import random
import time
from datetime import datetime
import requests

# ── Paths ─────────────────────────────────────────────────────
BIRD_CARD_DIR = DATA_DIR / "bird_card"
OUTPUT_JSON   = BIRD_CARD_DIR / "colombian_birds.json"
BIRD_CARD_DIR.mkdir(parents=True, exist_ok=True)

# ── Build species dict from df (already loaded above) ─────────
filtered_species = {}
counts = df["scientific_name"].value_counts()
for sci_name, count in counts.items():
    row = df[df["scientific_name"] == sci_name].iloc[0]
    filtered_species[sci_name] = {
        "english_name":     row.get("en", "") or "",
        "genus":            row.get("gen", "") or "",
        "species":          row.get("sp", "")  or "",
        "recording_count":  int(count),
    }

species_list = list(filtered_species.keys())
print(f"Species in filtered set : {len(species_list)}")

# ── Load existing output (resumable) ──────────────────────────
if OUTPUT_JSON.exists() and not FORCE_REFRESH:
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        existing = json.load(f)
    if isinstance(existing, dict):
        cards = existing.get("species", {})
    elif isinstance(existing, list):
        cards = {e["scientific_name"]: e for e in existing if "scientific_name" in e}
        print(f"  (converted legacy list → {len(cards)} entries)")
    else:
        cards = {}
    print(f"Resuming — {len(cards)} entries already saved.")
else:
    cards = {}

# ── Config ────────────────────────────────────────────────────
INAT_DELAY   = 1.5
GBIF_DELAY   = 1.0
WIKI_DELAY   = 1.5
MAX_RETRIES  = 5
BACKOFF_BASE = 2.0

HEADERS = {
    "User-Agent": "ColombianBirdCards/1.0 (academic research; contact: researcher@example.com)",
    "Accept":     "application/json",
}
CARD_SESSION = requests.Session()
CARD_SESSION.headers.update(HEADERS)

# ── IUCN lookup tables ────────────────────────────────────────
IUCN_STATUS_MAP = {
    "LC": {"code": "LC", "label_es": "Preocupación menor",         "label_en": "Least Concern"},
    "NT": {"code": "NT", "label_es": "Casi amenazada",              "label_en": "Near Threatened"},
    "VU": {"code": "VU", "label_es": "Vulnerable",                  "label_en": "Vulnerable"},
    "EN": {"code": "EN", "label_es": "En peligro",                  "label_en": "Endangered"},
    "CR": {"code": "CR", "label_es": "En peligro crítico",          "label_en": "Critically Endangered"},
    "EW": {"code": "EW", "label_es": "Extinta en estado silvestre", "label_en": "Extinct in the Wild"},
    "EX": {"code": "EX", "label_es": "Extinta",                     "label_en": "Extinct"},
    "DD": {"code": "DD", "label_es": "Datos insuficientes",         "label_en": "Data Deficient"},
    "NE": {"code": "NE", "label_es": "No evaluada",                 "label_en": "Not Evaluated"},
}
UNKNOWN_STATUS = IUCN_STATUS_MAP["NE"]

IUCN_QIDS = {
    "Q211005":  "LC",
    "Q719675":  "NT",
    "Q278113":  "VU",
    "Q11394":   "EN",
    "Q219127":  "CR",
    "Q239509":  "EW",
    "Q237350":  "EX",
    "Q3245245": "DD",
}

INAT_STATUS_NAME_MAP = {
    "least concern":         "LC",
    "near threatened":       "NT",
    "vulnerable":            "VU",
    "endangered":            "EN",
    "critically endangered": "CR",
    "extinct in the wild":   "EW",
    "extinct":               "EX",
    "data deficient":        "DD",
}

_wikidata_cache = {}

# ── Utilities ─────────────────────────────────────────────────
def safe_get(url, params=None, label=""):
    for attempt in range(MAX_RETRIES):
        try:
            r = CARD_SESSION.get(url, params=params, timeout=20)
            if r.status_code == 429:
                wait = BACKOFF_BASE ** (attempt + 2) + random.uniform(2, 5)
                print(f"\n  [429] {label} — retrying in {wait:.1f}s")
                time.sleep(wait)
                continue
            if r.status_code >= 500:
                wait = BACKOFF_BASE ** attempt + random.uniform(1, 3)
                print(f"\n  [{r.status_code}] {label} — retrying in {wait:.1f}s")
                time.sleep(wait)
                continue
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r
        except requests.exceptions.Timeout:
            time.sleep(BACKOFF_BASE ** attempt + 2)
        except requests.exceptions.ConnectionError:
            wait = BACKOFF_BASE ** attempt + random.uniform(1, 3)
            print(f"\n  [conn error] {label} — retrying in {wait:.1f}s ({attempt+1}/{MAX_RETRIES})")
            CARD_SESSION.close()
            CARD_SESSION.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            print(f"\n  [error] {label}: {e}")
            time.sleep(BACKOFF_BASE ** attempt)
    print(f"\n  [FAILED] {label}")
    return None


def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\[\[(?:[^\]|]+\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\{\{[^}]+\}\}', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def is_garbage_description(text):
    if not text:
        return True
    t = text.lower()
    if text.count("|") >= 2:
        return True
    if sum(1 for c in text if c.isupper()) / max(len(text), 1) > 0.3:
        return True
    garbage_patterns = [
        r"parque nacional natural", r"reserva forestal",
        r"área importante para la conservaci", r"ecorregi",
        r"iucn red list", r"least concern$", r"endangered$",
        r"vulnerable$", r"critically endangered$",
        r"\bcites\b", r"appendix [ivi]+", r"^[a-z ,;]+$",
    ]
    for pat in garbage_patterns:
        if re.search(pat, t):
            return True
    if "." not in text:
        return True
    bio_verbs = [
        "is ", "are ", "was ", "has ", "have ", "found ", "breed", "inhabit",
        "feed", "nest", "occur", "range", "measure", "weigh", "plumage",
        "species", "bird", "endemic", "distributed", "known",
        "es un", "es una", "ave ", "pájaro", "especie", "se encuentra",
        "habita", "distribuye", "mide ", "pesa ", "plumaje", "endémic",
        "conocid", "alimenta", "anida", "nidifica", "registr",
    ]
    return not any(v in t for v in bio_verbs)


def save_cards(cards_dict):
    out = {
        "generated_at":  datetime.utcnow().isoformat() + "Z",
        "total_species": len(cards_dict),
        "species":       cards_dict,
    }
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)


# ── Conservation status helpers ───────────────────────────────
def _iucn_from_code(raw_code):
    code = (raw_code or "").strip().upper()
    return IUCN_STATUS_MAP.get(code, UNKNOWN_STATUS)


def _iucn_from_inat_name(status_name):
    key = (status_name or "").strip().lower()
    code = INAT_STATUS_NAME_MAP.get(key)
    return IUCN_STATUS_MAP.get(code, UNKNOWN_STATUS) if code else None


def _wikidata_entity_by_site(page_title, site="enwiki"):
    cache_key = f"{site}:{page_title}"
    if cache_key in _wikidata_cache:
        return _wikidata_cache[cache_key]
    r = safe_get(
        "https://www.wikidata.org/w/api.php",
        params={
            "action": "wbgetentities", "sites": site, "titles": page_title,
            "props": "claims", "format": "json",
        },
        label=f"Wikidata [{site}:{page_title}]",
    )
    if r is None:
        return None
    for qid, entity in r.json().get("entities", {}).items():
        if not qid.startswith("-"):
            _wikidata_cache[cache_key] = entity
            return entity
    return None


def _status_from_wikidata_entity(entity):
    if not entity:
        return None
    claims = entity.get("claims", {})
    if "P141" not in claims:
        return None
    try:
        status_qid = claims["P141"][0]["mainsnak"]["datavalue"]["value"]["id"]
        code = IUCN_QIDS.get(status_qid)
        return IUCN_STATUS_MAP.get(code) if code else None
    except (KeyError, IndexError, TypeError):
        return None


def get_conservation_status(info_url, scientific_name, wiki_iucn_code=""):
    if wiki_iucn_code and wiki_iucn_code in IUCN_STATUS_MAP:
        return IUCN_STATUS_MAP[wiki_iucn_code]
    titles_to_try = []
    if info_url and "/wiki/" in info_url:
        raw  = requests.utils.unquote(
            info_url.rstrip("/").split("/wiki/")[-1]
        ).replace("_", " ")
        site = "eswiki" if "es.wikipedia.org" in info_url else "enwiki"
        titles_to_try.append((raw, site))
    if scientific_name:
        titles_to_try.append((scientific_name, "enwiki"))
        titles_to_try.append((scientific_name, "eswiki"))
    for title, site in titles_to_try:
        entity = _wikidata_entity_by_site(title, site)
        time.sleep(0.4)
        result = _status_from_wikidata_entity(entity)
        if result:
            return result
    if scientific_name:
        try:
            query = f'''
            SELECT ?status WHERE {{
              ?item wdt:P225 "{scientific_name}" .
              ?item wdt:P141 ?status .
            }} LIMIT 1
            '''
            r = safe_get(
                "https://query.wikidata.org/sparql",
                params={"query": query, "format": "json"},
                label=f"SPARQL status [{scientific_name}]",
            )
            time.sleep(0.8)
            if r:
                bindings = r.json().get("results", {}).get("bindings", [])
                if bindings:
                    qid  = bindings[0]["status"]["value"].split("/")[-1]
                    code = IUCN_QIDS.get(qid)
                    if code:
                        return IUCN_STATUS_MAP[code]
        except Exception as e:
            print(f"  [WARN SPARQL] {scientific_name}: {e}")
    return UNKNOWN_STATUS


# ── Data sources ──────────────────────────────────────────────
def inat_taxa(scientific_name):
    result = {"description": "", "image_url": "", "info_url": "", "conservation_status": None}
    r = safe_get(
        "https://api.inaturalist.org/v1/taxa",
        params={"q": scientific_name, "per_page": 1, "rank": "species"},
        label=f"iNat taxa [{scientific_name}]",
    )
    time.sleep(INAT_DELAY)
    if r is None:
        return result
    results = r.json().get("results", [])
    if not results:
        return result
    taxon = results[0]
    wiki_summary = clean_text(taxon.get("wikipedia_summary", "") or "")
    if wiki_summary:
        result["description"] = wiki_summary[:800]
    result["info_url"]  = taxon.get("wikipedia_url", "") or ""
    photo = taxon.get("default_photo") or {}
    result["image_url"] = photo.get("medium_url", "") or photo.get("url", "") or ""
    cs          = taxon.get("conservation_status") or {}
    raw_code    = cs.get("status") or ""
    status_name = taxon.get("conservation_status_name") or ""
    if raw_code:
        status = _iucn_from_code(raw_code)
        if status["code"] != "NE":
            result["conservation_status"] = status
    if result["conservation_status"] is None and status_name:
        result["conservation_status"] = _iucn_from_inat_name(status_name)
    return result


def inat_sightings(scientific_name):
    r = safe_get(
        "https://api.inaturalist.org/v1/observations",
        params={"taxon_name": scientific_name, "quality_grade": "research", "per_page": 0},
        label=f"iNat obs [{scientific_name}]",
    )
    time.sleep(INAT_DELAY)
    return int(r.json().get("total_results", 0)) if r else 0


def gbif_description(scientific_name):
    r = safe_get(
        "https://api.gbif.org/v1/species/match",
        params={"name": scientific_name, "kingdom": "Animalia", "class": "Aves"},
        label=f"GBIF match [{scientific_name}]",
    )
    time.sleep(GBIF_DELAY)
    if r is None:
        return ""
    data = r.json()
    key = data.get("usageKey") or data.get("speciesKey")
    if not key:
        return ""
    r2 = safe_get(
        f"https://api.gbif.org/v1/species/{key}/descriptions",
        label=f"GBIF desc [{scientific_name}]",
    )
    time.sleep(GBIF_DELAY)
    if r2 is None:
        return ""
    best_text, best_score = "", 0
    for d in r2.json().get("results", []):
        lang  = (d.get("language") or "").lower()
        dtype = (d.get("type") or "").lower()
        text  = clean_text(d.get("description") or "")
        if len(text) < 120 or is_garbage_description(text):
            continue
        score = 0
        if lang in ("spa", "es"):    score += 18
        elif lang in ("eng", "en"):  score += 12
        elif lang:                   score -= 5
        if dtype in ("", "general", "description", "biology", "ecology", "diagnostic"):
            score += 8
        if dtype in ("distribution", "conservation status", "threats", "habitat"):
            score -= 5
        score += min(len(text) // 80, 8)
        if score > best_score:
            best_score, best_text = score, text
    return best_text[:800]


def wiki_intro(scientific_name, lang="es"):
    base = f"https://{lang}.wikipedia.org/w/api.php"
    out  = {"description": "", "info_url": "", "image_url": ""}
    r = safe_get(
        base,
        params={"action": "query", "list": "search", "srsearch": scientific_name,
                "srlimit": 3, "format": "json"},
        label=f"Wiki {lang.upper()} search [{scientific_name}]",
    )
    time.sleep(WIKI_DELAY)
    if r is None:
        return out
    results = r.json().get("query", {}).get("search", [])
    if not results:
        return out
    genus   = scientific_name.split()[0].lower()
    epithet = scientific_name.split()[-1].lower()
    title   = next(
        (res["title"] for res in results
         if genus in res["title"].lower() or epithet in res["title"].lower()),
        results[0]["title"],
    )
    r2 = safe_get(
        base,
        params={"action": "query", "titles": title, "prop": "extracts|pageimages",
                "exintro": 1, "explaintext": 1, "pithumbsize": 600, "format": "json"},
        label=f"Wiki {lang.upper()} extract [{title}]",
    )
    time.sleep(WIKI_DELAY)
    safe_title      = title.replace(" ", "_")
    out["info_url"] = f"https://{lang}.wikipedia.org/wiki/{safe_title}"
    if r2 is None:
        return out
    for pg in r2.json().get("query", {}).get("pages", {}).values():
        if pg.get("pageid", -1) == -1:
            break
        extract = clean_text(pg.get("extract", "") or "")
        if extract and not is_garbage_description(extract):
            sentences = re.split(r'(?<=[.!?])\s+', extract)
            out["description"] = " ".join(sentences[:3])[:650]
        thumb = pg.get("thumbnail") or {}
        out["image_url"] = thumb.get("source", "")
    return out


def wiki_taxonomy(page_title):
    result = {"order": "", "family": "", "iucn_code": ""}
    if not page_title:
        return result
    is_es = False
    if page_title.startswith("http"):
        is_es      = "es.wikipedia.org" in page_title
        page_title = requests.utils.unquote(
            page_title.rstrip("/").split("/")[-1]
        ).replace("_", " ")
    apis = (
        ["https://es.wikipedia.org/w/api.php", "https://en.wikipedia.org/w/api.php"]
        if is_es else
        ["https://en.wikipedia.org/w/api.php", "https://es.wikipedia.org/w/api.php"]
    )
    def extract_field(wikitext, key):
        m = re.search(r'\|\s*' + key + r'\s*=\s*([^\|\}\n]+)', wikitext, re.IGNORECASE)
        if not m:
            return ""
        raw = m.group(1).strip()
        raw = re.sub(r'\[\[(?:[^\]|]+\|)?([^\]]+)\]\]', r'\1', raw)
        raw = re.sub(r'\{\{[^}]+\}\}', '', raw)
        raw = re.sub(r'<[^>]+>', '', raw)
        return raw.strip()
    for api_url in apis:
        r = safe_get(
            api_url,
            params={"action": "query", "titles": page_title, "prop": "revisions",
                    "rvprop": "content", "rvslots": "main", "rvlimit": 1, "format": "json"},
            label=f"Wiki taxobox [{page_title}]",
        )
        time.sleep(WIKI_DELAY)
        if r is None:
            continue
        wikitext = ""
        for pg in r.json().get("query", {}).get("pages", {}).values():
            if pg.get("pageid", -1) == -1:
                continue
            revs = pg.get("revisions", [])
            if revs:
                wikitext = revs[0].get("slots", {}).get("main", {}).get("*", "")
        if not wikitext:
            continue
        order  = extract_field(wikitext, "ordo")   or extract_field(wikitext, "order")
        family = extract_field(wikitext, "familia") or extract_field(wikitext, "family")
        if order or family:
            result["order"], result["family"] = order, family
        for field in ["estado_UICN", "status", "conservation_status"]:
            raw = extract_field(wikitext, field).upper()
            if raw in IUCN_STATUS_MAP:
                result["iucn_code"] = raw
                break
        if result["order"] or result["family"]:
            return result
    r = safe_get(
        "https://api.gbif.org/v1/species/match",
        params={"name": page_title, "kingdom": "Animalia", "class": "Aves"},
        label=f"GBIF taxonomy [{page_title}]",
    )
    time.sleep(GBIF_DELAY)
    if r:
        data = r.json()
        result["order"]  = data.get("order", "")  or ""
        result["family"] = data.get("family", "") or ""
        if result["order"] or result["family"]:
            return result
    print(f"  [WARN taxonomy] No order/family found for: {page_title}")
    return result


def build_taxonomy(sci_name, sp_info, extra):
    genus   = sp_info.get("genus", "")   or sci_name.split()[0]
    species = sp_info.get("species", "") or (sci_name.split()[1] if " " in sci_name else "")
    return {
        "kingdom": "Animalia",
        "phylum":  "Chordata",
        "class":   "Aves",
        "order":   extra.get("order",  ""),
        "family":  extra.get("family", ""),
        "genus":   genus,
        "species": species,
    }


def auto_description(sci_name, en_name, tax):
    parts = [f"{en_name} ({sci_name})" if en_name else sci_name, "is a bird species"]
    if tax.get("order"):
        parts.append(f"in the order {tax['order']}")
    if tax.get("family"):
        parts.append(f"(family {tax['family']})")
    parts.append("recorded in Colombia.")
    return " ".join(parts)


# ── Skip guard — all species already processed ────────────────
to_fetch = [s for s in species_list if s not in cards]

if not to_fetch:
    print(f"✓ Bird cards already complete ({len(cards)} species). Skipping fetch.")
else:
    print(f"Entries to fetch : {len(to_fetch)}  (skipping {len(cards)} already done)")
    print(f"Estimated time   : ~{len(to_fetch) * 10 // 60} min\n")

    # ── Main loop ──────────────────────────────────────────────
    for i, sci_name in enumerate(to_fetch, 1):
        sp_info   = filtered_species[sci_name]
        en_name   = sp_info.get("english_name", "") or ""
        rec_count = sp_info.get("recording_count", 0)

        print(f"[{i:>3}/{len(to_fetch)}] {sci_name:<38}", end=" ", flush=True)

        # 1. Wikipedia ES
        wiki_es     = wiki_intro(sci_name, lang="es")
        description = wiki_es["description"]
        image_url   = wiki_es["image_url"]
        info_url    = wiki_es["info_url"]
        desc_source = "Wikipedia ES" if description else ""

        # 2. iNaturalist
        inat = inat_taxa(sci_name)
        if not image_url:
            image_url = inat["image_url"]
        if not info_url:
            info_url = inat["info_url"]
        if not description and inat["description"]:
            description = inat["description"]
            desc_source = "iNat/Wikipedia EN"

        # 3. GBIF descriptions
        if not description:
            description = gbif_description(sci_name)
            if description:
                desc_source = "GBIF"

        # 4. Wikipedia EN (last resort)
        if not description or not image_url or not info_url:
            wiki_en = wiki_intro(sci_name, lang="en")
            if not description and wiki_en["description"]:
                description = wiki_en["description"]
                desc_source = "Wikipedia EN"
            if not image_url:
                image_url = wiki_en["image_url"]
            if not info_url:
                info_url = wiki_en["info_url"]

        # Taxonomy + IUCN code from taxobox
        taxo_extra = wiki_taxonomy(info_url or sci_name)
        tax        = build_taxonomy(sci_name, sp_info, taxo_extra)

        # iNaturalist sightings
        sightings = inat_sightings(sci_name)

        # Conservation status: iNat → taxobox → Wikidata → SPARQL
        conservation = inat.get("conservation_status")
        if not conservation or conservation["code"] == "NE":
            conservation = get_conservation_status(
                info_url, sci_name,
                wiki_iucn_code=taxo_extra.get("iucn_code", ""),
            )

        # Fallbacks — guarantee no nulls
        if not description:
            description = auto_description(sci_name, en_name, tax)
            desc_source = "auto"
        if not info_url:
            info_url = f"https://en.wikipedia.org/wiki/{sci_name.replace(' ', '_')}"
        if not image_url:
            r_img = safe_get(
                "https://api.inaturalist.org/v1/taxa/autocomplete",
                params={"q": sci_name, "per_page": 1},
                label=f"iNat img fallback [{sci_name}]",
            )
            time.sleep(INAT_DELAY)
            if r_img:
                res = r_img.json().get("results", [])
                if res:
                    p = res[0].get("default_photo") or {}
                    image_url = p.get("medium_url", "") or p.get("url", "")
            if not image_url:
                image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ac/No_image_available.svg/480px-No_image_available.svg.png"

        cards[sci_name] = {
            "scientific_name":       sci_name,
            "english_name":          en_name,
            "inat_sightings":        sightings,
            "xeno_canto_recordings": rec_count,
            "description":           description,
            "description_source":    desc_source,
            "conservation_status":   conservation,
            "taxonomy":              tax,
            "image_url":             image_url,
            "info_url":              info_url,
        }

        img_ok  = "img=OK" if image_url and "No_image" not in image_url else "img=fallback"
        cons_ok = conservation["code"] if conservation else "??"
        print(f"iNat={sightings:>6}  status={cons_ok:<3}  src={desc_source:<14}  {img_ok}")

        if i % 5 == 0:
            save_cards(cards)
            print(f"  >>> Saved {len(cards)} entries.\n")

        time.sleep(0.5)

    # ── Mystery Mystery entries ────────────────────────────────
    _mystery_desc = (
        "No se pudo determinar una especie específica para esta grabación, "
        "pero se confirmó que es un canto de ave registrado en Colombia."
    )
    _mystery_tax = {
        "kingdom": "Unknown", "phylum": "Unknown", "class": "Unknown",
        "order": "Unknown",   "family": "Unknown",
        "genus": "Unknown",   "species": "Unknown",
    }
    for mystery_key in ("Mystery Mystery", "Mystery mystery"):
        if mystery_key not in cards:
            cards[mystery_key] = {
                "scientific_name":       "Unknown / Uncatalogued Species",
                "english_name":          "Unknown / Uncatalogued Species",
                "inat_sightings":        0,
                "xeno_canto_recordings": 0,
                "description":           _mystery_desc,
                "description_source":    "manual",
                "conservation_status":   UNKNOWN_STATUS,
                "taxonomy":              _mystery_tax,
                "image_url":             "",
                "info_url":              "",
            }
            print(f"{mystery_key} entry added.")

    save_cards(cards)
    print(f"\nDone. {len(cards)} entries saved to:\n  {OUTPUT_JSON}")

    # ── Integrity check ───────────────────────────────────────
    null_desc = [k for k, v in cards.items() if not v.get("description")]
    null_img  = [k for k, v in cards.items() if not v.get("image_url")]
    null_url  = [k for k, v in cards.items() if not v.get("info_url")]
    ne_status = [k for k, v in cards.items() if v.get("conservation_status", {}).get("code") == "NE"]
    by_source = {}
    for v in cards.values():
        s = v.get("description_source", "unknown")
        by_source[s] = by_source.get(s, 0) + 1
    print(f"   Null descriptions : {len(null_desc)}")
    print(f"   Null images       : {len(null_img)}")
    print(f"   Null info URLs    : {len(null_url)}")
    print(f"   NE status (may be real): {len(ne_status)}")
    print(f"   Description sources: {by_source}")
    for k in set(null_desc + null_img + null_url):
        print(f"   WARNING missing data: {k}")

## Block 2 — Audio Download

Downloads MP3 files from Xeno-Canto asynchronously with rate-limiting, retries, and resume support. Scheme-less URLs (`//xeno-canto.org/...`) are normalised to `https:` automatically.

In [ ]:
# =============================================================
# BLOQUE 2: Descarga de audios (asyncio + retrys)
# Depends on the configuration block
# =============================================================

import asyncio
import logging
import aiohttp
import nest_asyncio
import pandas as pd
from pathlib import Path
from tqdm.asyncio import tqdm as atqdm

nest_asyncio.apply()

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

# ── Download parameters ────────────────────────────────────

MAX_CONCURRENT = 5      # concurrent connections (be polite to XC)
MAX_RETRIES    = 3      # retries per file
MIN_FILE_SIZE  = 1024   # minimum bytes to consider a file valid

# ── Load filtered metadata if not already in memory ────────────


df_filtered = pd.read_parquet(FILTERED_PARQUET_PATH)
log.info(f"Loaded {len(df_filtered):,} records from {FILTERED_PARQUET_PATH}")

# ── Utilities ────────────────────────────────────────────────

def _build_url(rec: dict) -> str:
    """
    The Xeno-Canto 'file' field is scheme-less: '//xeno-canto.org/...'
    This function normalises it into a downloadable URL.
    """
    url = rec.get("file", "")
    if url:
        if url.startswith("//"):
            return "https:" + url       # most common case
        if url.startswith("http"):
            return url                  # already complete
    # Fallback only if file field is genuinely empty
    return f"https://xeno-canto.org/{rec.get('id', '')}/download"

def _audio_filename(rec: dict) -> str:
    species = (
        rec.get("en", "unknown")
        .replace(" ", "_")
        .replace("/", "-")
    )
    return f"XC{rec.get('id', 'X')}_{species}.mp3"

# ── Single-file download with retries ─────────────────────

async def _download_one(
    session:  aiohttp.ClientSession,
    sem:      asyncio.Semaphore,
    rec:      dict,
    dest_dir: Path,
) -> Path | None:

    url  = _build_url(rec)
    dest = dest_dir / _audio_filename(rec)

    # Skip if already downloaded and valid
    if dest.exists() and dest.stat().st_size > MIN_FILE_SIZE:
        return dest

    async with sem:
        for attempt in range(MAX_RETRIES):
            try:
                timeout = aiohttp.ClientTimeout(total=60)
                async with session.get(url, timeout=timeout) as r:

                    if r.status == 429:                         # rate-limited
                        wait = 2 ** attempt * 5
                        log.warning(f"429 rate limit — waiting {wait}s")
                        await asyncio.sleep(wait)
                        continue

                    if r.status == 503:                         # service unavailable
                        wait = 2 ** attempt * 3
                        log.warning(
                            f"503 en XC{rec.get('id')} — "
                            f"retry {attempt + 1}/{MAX_RETRIES} en {wait}s"
                        )
                        await asyncio.sleep(wait)
                        continue

                    r.raise_for_status()
                    content = await r.read()

                    if len(content) < MIN_FILE_SIZE:
                        log.warning(
                            f"Archivo demasiado pequeño "
                            f"({len(content)}B) — XC{rec.get('id')}"
                        )
                        return None

                    dest.write_bytes(content)
                    return dest

            except asyncio.TimeoutError:
                log.warning(
                    f"Timeout XC{rec.get('id')} — "
                    f"retry {attempt + 1}/{MAX_RETRIES}"
                )
                await asyncio.sleep(2 ** attempt)

            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    log.warning(f"Final failure XC{rec.get('id')}: {e}")
                    return None
                await asyncio.sleep(2 ** attempt)

    return None

# ── Async orchestrator ─────────────────────────────────────

async def _download_all_async(
    records:  list[dict],
    dest_dir: Path,
) -> dict:

    dest_dir.mkdir(parents=True, exist_ok=True)
    sem   = asyncio.Semaphore(MAX_CONCURRENT)
    paths = {}

    connector = aiohttp.TCPConnector(
        limit=20,
        limit_per_host=MAX_CONCURRENT
    )
    headers = {"User-Agent": "BirdDatasetBuilder/1.0"}

    async with aiohttp.ClientSession(
        connector=connector,
        headers=headers
    ) as session:

        tasks = {
            rec["id"]: asyncio.create_task(
                _download_one(session, sem, rec, dest_dir)
            )
            for rec in records
        }

        for xc_id, task in atqdm(
            tasks.items(),
            total=len(tasks),
            desc="Downloading audio"
        ):
            try:
                paths[xc_id] = await task
            except Exception as e:
                log.error(f"Unexpected error on XC{xc_id}: {e}")
                paths[xc_id] = None

    return paths

# ── Main function ─────────────────────────────────────────

def download_all_audio(df: pd.DataFrame, dest_dir: Path) -> pd.DataFrame:
    records = df.to_dict("records")
    paths   = asyncio.run(_download_all_async(records, dest_dir))
    df = df.copy()
    df["local_path"] = df["id"].map(paths)
    return df

# ── Execution ─────────────────────────────────────────────────

log.info("=" * 60)
log.info("BLOCK 2 — Audio download (asyncio)")
log.info("=" * 60)
log.info(f"Destination     : {AUDIO_DIR}")
log.info(f"Concurrent  : {MAX_CONCURRENT}")
log.info(f"Retries  : {MAX_RETRIES}")
log.info(f"Total       : {len(df_filtered):,} grabaciones")

# Diagnóstico de URLs en 3 registros antes de lanzar todo
log.info("URL sample:")
for rec in df_filtered.head(3).to_dict("records"):
    log.info(f"  XC{rec['id']} → {_build_url(rec)}")

df_with_audio = download_all_audio(df_filtered, dest_dir=AUDIO_DIR)

ok    = df_with_audio["local_path"].notna().sum()
total = len(df_with_audio)
log.info(f"Successfully downloaded : {ok:,} / {total:,}")
log.info(f"Failed                 : {total - ok:,}")
log.info(f"Audio saved to      : {AUDIO_DIR}")

In [ ]:
import imageio_ffmpeg
ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print(f"ffmpeg found at: {ffmpeg_path}")

# Add to PATH for this session
import os
os.environ["PATH"] += os.pathsep + str(Path(ffmpeg_path).parent)
print("PATH updated")

# Verify pydub can find it
from pydub import AudioSegment
AudioSegment.converter = ffmpeg_path
print("pydub configured with ffmpeg")

## Block 3 — Audio Cleaning (VAD + DSP)

Removes human speech using Silero VAD and applies a DSP pipeline: Butterworth band-pass filter (1–10 kHz), spectral subtraction, RMS silence removal, and peak normalisation.

Key calibration parameters:
- `SPEECH_THRESHOLD = 0.85` — high-confidence threshold to avoid removing bird calls classified as speech (default Silero threshold 0.5 is tuned for human voice)
- `N_CONSEC = 4` — requires 4 consecutive 32 ms windows (~128 ms) of speech before a region is removed
- `DRY_RUN = True` — diagnose parameter settings without writing any files

In [ ]:
# =============================================================
# BLOCK 3 v5 — Audio cleaning with calibrated VAD
#
# Changes vs v4:
#   1. SPEECH_THRESHOLD: 0.5 → 0.85
#      Silero was trained on human voice. At 0.5, harmonic
#      calls (wrens, tanagers) get classified as speech.
#      0.85 requires high confidence before removing audio.
#
#   2. Temporal mask smoothing (morphological dilation)
#      A single 32 ms window is not removed in isolation.
#      N_CONSEC consecutive speech windows are required
#      before marking a region. Prevents fragmentation of short calls.
#
#   3. Automatic per-species metrics
#      Without listening: reports % duration retained
#      per species, VAD probability distribution, and a warning if
#      any species loses >50 % of mean duration.
#
#   4. Modo diagnostic run (DRY_RUN=True)
#      Processes N_DIAG files and generates a report without writing
#      files. Useful for calibrating before a full run.
# =============================================================

import asyncio
import logging
import warnings
import multiprocessing as _mp
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
import librosa
import soundfile as sf
import pandas as pd
from scipy.signal import butter, filtfilt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

# =============================================================
# Directorios (ajusta según tu PROJECT_DIR)
# =============================================================

# Uncomment the line that matches your environment:
# from pathlib import Path; PROJECT_DIR = Path.cwd() / "bird_project"
CLEAN_DIR = PROCESSED_DIR / "clean_audio"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =============================================================
# Audio parameters
# =============================================================

SR_LOAD          = 16000
SR_OUT           = 32000
F_MIN            = 1000.0
F_MAX            = 10000.0
N_FFT            = 1024
HOP              = 256
SILENCE_DB       = -40.0
MIN_BIRD_SEGMENT = 0.5     # minimum seconds of useful signal
ALPHA            = 2.0     # spectral subtraction: aggressiveness
BETA             = 0.01    # spectral subtraction: minimum floor

# =============================================================
# VAD parameters — key calibration values
# =============================================================

# WAS: 0.5 (default threshold for human voice)
# NOW: 0.85 — requires high confidence to classify as speech
# Recommended range for bird calls: 0.80–0.92
# Raise if too much bird audio is lost; lower if human speech remains
SPEECH_THRESHOLD = 0.85

# Consecutive windows required to confirm speech.
# Silero: 512 samples @ 16 kHz = 32 ms per window.
# N_CONSEC=4 → requires 128 ms of continuous speech to remove.
# Prevents a short 32–64 ms bird call from being removed due to a
# window with a slightly elevated probability.
N_CONSEC = 4

VAD_BATCH_SIZE = 64
N_LOAD_WORKERS = 8
N_DSP_WORKERS  = max(1, _mp.cpu_count() - 2)

# =============================================================
# Diagnostic mode
# =============================================================

# DRY_RUN=True: processes N_DIAG files without writing any output.
# Generates a metrics report only — ideal for calibration.
DRY_RUN = False
N_DIAG  = 200   # archivos a analizar en modo diagnostic run


# =============================================================
# 1. Audio loading
# =============================================================

def _load_audio(filepath: Path) -> np.ndarray | None:
    try:
        y, _ = librosa.load(str(filepath), sr=SR_LOAD, mono=True)
        if len(y) > 0:
            return y.astype(np.float32)
    except Exception:
        pass
    try:
        import imageio_ffmpeg
        from pydub import AudioSegment
        AudioSegment.converter = imageio_ffmpeg.get_ffmpeg_exe()
        seg = AudioSegment.from_file(str(filepath))
        seg = seg.set_channels(1).set_frame_rate(SR_LOAD)
        y   = np.array(seg.get_array_of_samples(), dtype=np.float32)
        y  /= np.iinfo(seg.array_type).max
        return y
    except Exception:
        pass
    try:
        y, sr = sf.read(str(filepath), dtype="float32", always_2d=False)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != SR_LOAD:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR_LOAD)
        return y.astype(np.float32)
    except Exception:
        return None


def _load_batch_parallel(filepaths: list[Path]) -> list[tuple[Path, np.ndarray | None]]:
    results = [None] * len(filepaths)

    def _load_indexed(args):
        idx, fp = args
        return idx, _load_audio(fp)

    with ThreadPoolExecutor(max_workers=N_LOAD_WORKERS) as ex:
        futures = {ex.submit(_load_indexed, (i, fp)): i
                   for i, fp in enumerate(filepaths)}
        for future in as_completed(futures):
            idx, y = future.result()
            results[idx] = (filepaths[idx], y)

    return results


# =============================================================
# 2. Silero VAD — calibrated threshold and temporal smoothing
# =============================================================

def _load_vad_model():
    from torch.hub import load as hub_load
    model, utils = hub_load(
        "snakers4/silero-vad",
        "silero_vad",
        force_reload=False,
        trust_repo=True,
    )
    model = model.to(DEVICE).eval()
    log.info(f"Silero VAD en: {next(model.parameters()).device}")
    if DEVICE == "cuda":
        with torch.no_grad():
            dummy = torch.zeros(1, 512, dtype=torch.float32, device=DEVICE)
            try:
                _ = model(dummy, SR_LOAD)
            except Exception:
                pass
        torch.cuda.empty_cache()
        log.info(f"VRAM tras warm-up: {torch.cuda.memory_allocated()/1024**2:.1f} MB")
    return model, utils[0]


def _dilate_mask(mask: np.ndarray, n_consec: int) -> np.ndarray:
    """
    Temporal smoothing: only marks a window as speech if
    there are at least n_consec consecutive windows marked.

    Ejemplo con n_consec=4:
      mask original: [0,0,1,0,1,1,1,1,0,0]
      mask suavizada: [0,0,0,0,1,1,1,1,0,0]
      → la ventana aislada en pos 2 y el par 4-5 no se eliminan.
    """
    if n_consec <= 1:
        return mask

    result = np.zeros_like(mask)
    n = len(mask)
    i = 0
    while i < n:
        if mask[i]:
            # count run of consecutive True values
            j = i
            while j < n and mask[j]:
                j += 1
            # only mark if run is long enough
            if (j - i) >= n_consec:
                result[i:j] = True
            i = j
        else:
            i += 1
    return result


@torch.no_grad()
def _vad_batch(
    audio_list: list[np.ndarray],
    vad_model,
    get_speech_timestamps,
    collect_probs: bool = False,
) -> tuple[list[np.ndarray], list[dict]]:
    """
    Returns (cleaned_audio_list, per_audio_metrics).
    Metrics include: dur_original, dur_after, pct_kept, prob_mean, prob_max.
    """
    WINDOW  = 512
    cleaned = []
    metrics = []

    for y in audio_list:
        n      = len(y)
        n_wins = n // WINDOW
        dur_orig = n / SR_LOAD

        if n_wins == 0:
            cleaned.append(y)
            metrics.append({
                "dur_before": dur_orig, "dur_after": dur_orig,
                "pct_kept": 100.0, "prob_mean": 0.0, "prob_max": 0.0,
                "fallback": False,
            })
            continue

        y_trim  = y[:n_wins * WINDOW]
        windows = torch.tensor(
            y_trim.reshape(n_wins, WINDOW),
            dtype=torch.float32,
            device=DEVICE,
        )

        try:
            probs = vad_model(windows, SR_LOAD).cpu().numpy()
        except Exception as e:
            log.debug(f"VAD GPU failed: {e} — fallback")
            cleaned.append(y)
            metrics.append({
                "dur_before": dur_orig, "dur_after": dur_orig,
                "pct_kept": 100.0, "prob_mean": float(np.nan), "prob_max": float(np.nan),
                "fallback": True,
            })
            continue
        finally:
            del windows
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        # Binary mask per window (calibrated threshold)
        raw_mask = probs > SPEECH_THRESHOLD          # shape: (n_wins,)

        # Temporal smoothing: remove short speech bursts
        smooth_mask = _dilate_mask(raw_mask, N_CONSEC)

        # Expand to sample level
        speech_mask = np.zeros(n, dtype=bool)
        for i, is_speech in enumerate(smooth_mask):
            if is_speech:
                start = i * WINDOW
                end   = min(start + WINDOW, n)
                speech_mask[start:end] = True

        bird = y[~speech_mask]
        dur_after = len(bird) / SR_LOAD

        if len(bird) >= MIN_BIRD_SEGMENT * SR_LOAD:
            cleaned.append(bird.astype(np.float32))
            pct = dur_after / dur_orig * 100 if dur_orig > 0 else 100.0
            fallback = False
        else:
            # Audio demasiado corto tras VAD: conservar original
            cleaned.append(y)
            dur_after = dur_orig
            pct = 100.0
            fallback = True

        metrics.append({
            "dur_before": round(dur_orig, 2),
            "dur_after":  round(dur_after, 2),
            "pct_kept":   round(pct, 1),
            "prob_mean":  round(float(probs.mean()), 4),
            "prob_max":   round(float(probs.max()),  4),
            "fallback":   fallback,
        })

    return cleaned, metrics


# =============================================================
# 3. DSP pipeline
# =============================================================

def _dsp_pipeline(y: np.ndarray) -> np.ndarray:
    y = librosa.resample(y, orig_sr=SR_LOAD, target_sr=SR_OUT)
    nyq  = SR_OUT / 2.0
    b, a = butter(4, [max(F_MIN/nyq, 1e-4), min(F_MAX/nyq, 0.9999)], btype="band")
    y    = filtfilt(b, a, y).astype(np.float32)

    S     = librosa.stft(y, n_fft=N_FFT, hop_length=HOP, window="hann")
    mag   = np.abs(S)
    phase = np.angle(S)
    n_nf  = max(1, mag.shape[1] // 10)
    n_idx = np.argsort(mag.sum(axis=0))[:n_nf]
    n_prof= mag[:, n_idx].mean(axis=1, keepdims=True)
    mag_c = np.maximum(mag - ALPHA * n_prof, BETA * n_prof)
    y_c   = librosa.istft(mag_c * np.exp(1j * phase), hop_length=HOP, window="hann")
    if len(y_c) < len(y):
        y_c = np.pad(y_c, (0, len(y) - len(y_c)))
    y = y_c[:len(y)].astype(np.float32)

    rms    = librosa.feature.rms(y=y, frame_length=1024, hop_length=HOP)[0]
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    mask   = rms_db > SILENCE_DB
    min_f  = int(MIN_BIRD_SEGMENT * SR_OUT / HOP)
    segs, i = [], 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            if (j - i) >= min_f:
                s = librosa.frames_to_samples(i, hop_length=HOP)
                e = librosa.frames_to_samples(j, hop_length=HOP)
                segs.append(y[s:e])
            i = j
        else:
            i += 1
    if segs:
        y = np.concatenate(segs).astype(np.float32)

    peak = np.max(np.abs(y))
    if peak > 1e-8:
        y = y / peak

    return y.astype(np.float32)


def _dsp_pipeline_measured(y: np.ndarray) -> dict:
    """
    Same as _dsp_pipeline but returns duration after each stage.
    Used in DRY_RUN to identify which stage removes the most audio.
    """
    sr = SR_LOAD  # audio enters at 16 kHz from VAD
    dur_in = len(y) / sr

    # 1. Resample
    y = librosa.resample(y, orig_sr=SR_LOAD, target_sr=SR_OUT)
    sr = SR_OUT
    dur_resample = len(y) / sr  # should equal dur_in

    # 2. Butterworth band-pass filter 1–10 kHz
    nyq  = SR_OUT / 2.0
    b, a = butter(4, [max(F_MIN/nyq, 1e-4), min(F_MAX/nyq, 0.9999)], btype="band")
    y    = filtfilt(b, a, y).astype(np.float32)
    dur_butter = len(y) / sr   # length unchanged

    # 3. Spectral subtraction
    S     = librosa.stft(y, n_fft=N_FFT, hop_length=HOP, window="hann")
    mag   = np.abs(S)
    phase = np.angle(S)
    n_nf  = max(1, mag.shape[1] // 10)
    n_idx = np.argsort(mag.sum(axis=0))[:n_nf]
    n_prof= mag[:, n_idx].mean(axis=1, keepdims=True)
    mag_c = np.maximum(mag - ALPHA * n_prof, BETA * n_prof)
    y_c   = librosa.istft(mag_c * np.exp(1j * phase), hop_length=HOP, window="hann")
    if len(y_c) < len(y):
        y_c = np.pad(y_c, (0, len(y) - len(y_c)))
    y = y_c[:len(y)].astype(np.float32)
    dur_spectral = len(y) / sr  # length unchanged

    # 4. RMS silence removal
    rms    = librosa.feature.rms(y=y, frame_length=1024, hop_length=HOP)[0]
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    mask   = rms_db > SILENCE_DB
    min_f  = int(MIN_BIRD_SEGMENT * SR_OUT / HOP)
    segs, i = [], 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            if (j - i) >= min_f:
                s = librosa.frames_to_samples(i, hop_length=HOP)
                e = librosa.frames_to_samples(j, hop_length=HOP)
                segs.append(y[s:e])
            i = j
        else:
            i += 1
    if segs:
        y = np.concatenate(segs).astype(np.float32)
    dur_rms = len(y) / sr

    return {
        "dur_in":       round(dur_in, 3),
        "dur_spectral": round(dur_spectral, 3),  # tras sustracción (igual a dur_in)
        "dur_rms":      round(dur_rms, 3),        # after silence removal ← main source of loss
        "pct_spectral": 100.0,                    # subtraction does not change length
        "pct_rms":      round(dur_rms / dur_in * 100, 1) if dur_in > 0 else 100.0,
    }


def _dsp_and_save(y: np.ndarray, out_path: Path, dry_run: bool = False) -> tuple[bool, dict]:
    """Returns (ok, stage_metrics). stage_metrics is only populated in dry_run."""
    try:
        if dry_run:
            stage = _dsp_pipeline_measured(y)
            return True, stage
        else:
            y_clean = _dsp_pipeline(y)
            sf.write(str(out_path), y_clean, SR_OUT, subtype="PCM_16")
            return True, {}
    except Exception as e:
        log.debug(f"DSP failed on {out_path.name}: {e}")
        return False, {}


# =============================================================
# 4. Automatic per-species metrics
# =============================================================

def compute_vad_metrics(
    df_metrics: pd.DataFrame,
    df_filtered: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merges VAD metrics with df_filtered to produce
    per-species statistics. Emits a WARNING if any species
    loses more than 50 % of mean duration.
    """
    df_m = df_metrics.copy()
    df_m["xc_id"] = df_m["filename"].str.extract(r"XC(\d+)")

    df_f = df_filtered[["id", "scientific_name", "en"]].copy()
    df_f["xc_id"] = df_f["id"].astype(str)

    merged = df_m.merge(df_f, on="xc_id", how="left")

    by_species = (
        merged.groupby("scientific_name")
        .agg(
            n_files       = ("xc_id",      "count"),
            dur_antes_med = ("dur_before",  "mean"),
            dur_despues_med=("dur_after",   "mean"),
            pct_kept_med  = ("pct_kept",    "mean"),
            pct_kept_min  = ("pct_kept",    "min"),
            prob_vad_med  = ("prob_mean",   "mean"),
            prob_vad_max  = ("prob_max",    "max"),
            n_fallback    = ("fallback",    "sum"),
        )
        .round(2)
        .sort_values("pct_kept_med")
    )

    # Automatic warnings
    alarmas = by_species[by_species["pct_kept_med"] < 50.0]
    if not alarmas.empty:
        log.warning(
            f"\n{'='*60}\n"
            f"⚠  {len(alarmas)} especies con <50% duration retained:\n"
            f"{alarmas[['n_files','pct_kept_med','prob_vad_med']].to_string()}\n"
            f"{'='*60}\n"
            f"→ Consider raising SPEECH_THRESHOLD (actual: {SPEECH_THRESHOLD})\n"
            f"  or increasing N_CONSEC (actual: {N_CONSEC})"
        )
    else:
        log.info(f"✓ All species retain ≥50 % of duration. "
                 f"SPEECH_THRESHOLD={SPEECH_THRESHOLD} looks appropriate.")

    return by_species


# =============================================================
# 5. Main pipeline
# =============================================================

def process_all(
    audio_dir:  Path,
    clean_dir:  Path,
    df_filtered: pd.DataFrame,
    dry_run:    bool = False,
    n_diag:     int  = 200,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (df_results, df_species_metrics).
    In dry_run=True mode no files are written; only n_diag files are processed.
    """
    mp3_files = sorted(audio_dir.glob("*.mp3"))

    if dry_run:
        # Tomar muestra estratificada per species para el diagnostic run
        # Extract xc_id from each MP3 and cross-reference with df_filtered
        id_to_species = dict(zip(
            df_filtered["id"].astype(str),
            df_filtered["scientific_name"]
        ))
        annotated = []
        for fp in mp3_files:
            xc_id = fp.stem.split("_")[0].replace("XC", "")
            sp = id_to_species.get(xc_id, "unknown")
            annotated.append((fp, sp))

        # Stratified sample: distribute n_diag proportionally
        # across species, guaranteeing at least 1 per species and
        # respecting the exact total limit.
        from collections import defaultdict
        import math

        per_species = defaultdict(list)
        for fp, sp in annotated:
            per_species[sp].append(fp)

        n_sp   = len(per_species)
        # Rounds: first 1 per species, then 2, etc., up to n_diag
        quota  = {sp: 0 for sp in per_species}
        total  = 0
        round_ = 1
        while total < n_diag:
            added = 0
            for sp, files in per_species.items():
                if quota[sp] < len(files) and quota[sp] < round_:
                    quota[sp] += 1
                    total += 1
                    added += 1
                    if total >= n_diag:
                        break
            if added == 0:
                break  # no more files available
            round_ += 1

        sampled = []
        for sp, n_take in quota.items():
            sampled.extend(per_species[sp][:n_take])

        mp3_files = sampled[:n_diag]
        log.info(f"[DRY RUN] Processing {len(mp3_files)} archivos "
                 f"({n_sp} species, ~{len(mp3_files)/max(n_sp,1):.1f} per species)")

    pending, results = [], []
    for fp in mp3_files:
        out = clean_dir / (fp.stem + "_clean.wav")
        if not dry_run and out.exists():
            results.append({
                "filename": fp.name, "original": fp,
                "clean": out, "ok": True,
                "dur_before": None, "dur_after": None,
                "pct_kept": None, "prob_mean": None,
                "prob_max": None, "fallback": False,
            })
        else:
            pending.append(fp)

    log.info(f"Total files : {len(mp3_files):,}")
    log.info(f"Already processed  : {len(results):,}")
    log.info(f"Pending     : {len(pending):,}")

    if not pending:
        return pd.DataFrame(results), pd.DataFrame()

    vad_model, get_speech_timestamps = _load_vad_model()
    LOAD_BATCH = VAD_BATCH_SIZE * 2

    all_metrics = []
    _done, _total = 0, len(pending)

    def _log_progress():
        pct = _done / _total * 100
        print(
            f"\r  {_done:,}/{_total:,} ({pct:.1f}%)  "
            f"{'█' * int(pct/5)}{'░' * (20 - int(pct/5))}",
            end="", flush=True,
        )

    print(f"\nStarting {'diagnostic run' if dry_run else 'processing'} "
          f"de {_total:,} files...\n")

    with ThreadPoolExecutor(max_workers=N_DSP_WORKERS) as dsp_pool:
        for batch_start in range(0, len(pending), LOAD_BATCH):
            batch_files = pending[batch_start: batch_start + LOAD_BATCH]
            loaded = _load_batch_parallel(batch_files)
            valid  = [(fp, y) for fp, y in loaded if y is not None]
            failed = [fp      for fp, y in loaded if y is None]

            for fp in failed:
                results.append({
                    "filename": fp.name, "original": fp,
                    "clean": None, "ok": False,
                    "dur_before": None, "dur_after": None,
                    "pct_kept": None, "prob_mean": None,
                    "prob_max": None, "fallback": False,
                })
                _done += 1
                _log_progress()

            if not valid:
                continue

            fps_v = [fp for fp, _ in valid]
            ys    = [y  for _,  y in valid]

            # VAD in GPU batches
            ys_clean_all, mets_all = [], []
            for i in range(0, len(ys), VAD_BATCH_SIZE):
                batch_y = ys[i:i + VAD_BATCH_SIZE]
                cleaned_batch, mets_batch = _vad_batch(
                    batch_y, vad_model, get_speech_timestamps,
                )
                ys_clean_all.extend(cleaned_batch)
                mets_all.extend(mets_batch)

            # Record VAD metrics
            for fp, met in zip(fps_v, mets_all):
                all_metrics.append({"filename": fp.name, **met})

            # DSP in threads
            dsp_futures = {}
            for fp, y in zip(fps_v, ys_clean_all):
                out_path = clean_dir / (fp.stem + "_clean.wav")
                fut = dsp_pool.submit(_dsp_and_save, y, out_path, dry_run)
                dsp_futures[fut] = (fp, out_path)

            for future in as_completed(dsp_futures):
                fp, out_path = dsp_futures[future]
                ok, stage = future.result()
                met = next(
                    (m for m in all_metrics if m["filename"] == fp.name), {}
                )
                results.append({
                    "filename":     fp.name,
                    "original":     fp,
                    "clean":        out_path if (ok and not dry_run) else None,
                    "ok":           ok,
                    "dur_before":   met.get("dur_before"),
                    "dur_after":    met.get("dur_after"),
                    "pct_kept":     met.get("pct_kept"),
                    "prob_mean":    met.get("prob_mean"),
                    "prob_max":     met.get("prob_max"),
                    "fallback":     met.get("fallback", False),
                    # Etapas DSP (only en dry_run)
                    "dsp_pct_rms":  stage.get("pct_rms"),
                    "dsp_dur_rms":  stage.get("dur_rms"),
                })
                _done += 1
                _log_progress()

    print("\n\nFinalizado.")
    df_results  = pd.DataFrame(results)
    df_metrics  = pd.DataFrame(all_metrics)

    # Cruzar scientific_name al df_results (útil para el reporte por etapa DSP)
    if not df_results.empty and not df_filtered.empty:
        id_map = dict(zip(df_filtered["id"].astype(str), df_filtered["scientific_name"]))
        df_results["xc_id"] = df_results["filename"].str.extract(r"XC(\d+)")
        df_results["scientific_name"] = df_results["xc_id"].map(id_map)

    df_species  = compute_vad_metrics(df_metrics, df_filtered) if not df_metrics.empty else pd.DataFrame()

    return df_results, df_species


# =============================================================
# 6. Ejecución
# =============================================================

log.info("=" * 60)
log.info("BLOQUE 3 v5 — VAD calibrado + métricas automáticas")
log.info("=" * 60)
log.info(f"Entrada          : {AUDIO_DIR}")
log.info(f"Salida           : {CLEAN_DIR}")
log.info(f"Dispositivo VAD  : {DEVICE.upper()}")
log.info(f"SPEECH_THRESHOLD : {SPEECH_THRESHOLD}  (v4: 0.5)")
log.info(f"N_CONSEC         : {N_CONSEC} ventanas ({N_CONSEC*32} ms mín para eliminar)")
log.info(f"VAD batch size   : {VAD_BATCH_SIZE}")
log.info(f"DSP workers      : {N_DSP_WORKERS}")
log.info(f"DRY_RUN          : {DRY_RUN}")

if "df_filtered" not in dir():
    df_filtered = pd.read_parquet(FILTERED_PARQUET_PATH)

df_clean, df_species = process_all(
    AUDIO_DIR, CLEAN_DIR, df_filtered,
    dry_run=DRY_RUN,
    n_diag=N_DIAG,
)

# =============================================================
# 7. Resumen de resultados
# =============================================================

ok    = df_clean["ok"].sum()
total = len(df_clean)

print(f"\n{'─'*60}")
print(f"{'[DRY RUN] ' if DRY_RUN else ''}Processed OK  : {ok:,} / {total:,}")
print(f"Failed         : {total - ok:,}")

if "dur_before" in df_clean.columns and df_clean["dur_before"].notna().any():
    d = df_clean.dropna(subset=["dur_before", "dur_after"])
    print(f"\nDuration BEFORE (s) — media: {d['dur_before'].mean():.1f}")
    print(f"Duration AFTER(s) — media: {d['dur_after'].mean():.1f}")
    print(f"% duration retained — media: {d['pct_kept'].mean():.1f}%")
    print(f"% duration retained — mediana: {d['pct_kept'].median():.1f}%")
    print(f"Files with fallback  : {d['fallback'].sum():,}  "
          f"({d['fallback'].mean()*100:.1f}% — kept original audio)")

    # VAD probability distribution (calibration indicator)
    print(f"\nMean VAD prob  : {d['prob_mean'].mean():.3f}")
    print(f"Max VAD prob    : {d['prob_max'].max():.3f}")
    print(f"Files with prob_max > {SPEECH_THRESHOLD:.2f}: "
          f"{(d['prob_max'] > SPEECH_THRESHOLD).sum():,} "
          f"({(d['prob_max'] > SPEECH_THRESHOLD).mean()*100:.1f}%)")

if not df_species.empty:
    print(f"\n{'─'*60}")
    print("Top 15 species with LEAST duration retained:")
    print(df_species.head(15)[
        ["n_files", "pct_kept_med", "pct_kept_min", "prob_vad_med", "n_fallback"]
    ].to_string())

    print(f"\n{'─'*60}")
    print("Top 15 species with MOST duration retained:")
    print(df_species.tail(15)[
        ["n_files", "pct_kept_med", "pct_kept_min", "prob_vad_med", "n_fallback"]
    ].to_string())

if DRY_RUN:
    print(f"\n{'─'*60}")
    print("DRY RUN mode — no files written.")

    # Desglose por etapa DSP
    d = df_clean.dropna(subset=["dsp_pct_rms"])
    if not d.empty:
        print(f"\n{'─'*60}")
        print("Desglose de pérdida por etapa DSP:")
        print(f"  VAD (speech removal) — mean retained: "
              f"{df_clean['pct_kept'].mean():.1f}%  "
              f"[only {(df_clean['prob_max'] > SPEECH_THRESHOLD).sum()} files affected]")
        print(f"  RMS silence removal — mean retained: "
              f"{d['dsp_pct_rms'].mean():.1f}%  ← main source of loss")
        print(f"  Mean duration after RMS : {d['dsp_dur_rms'].mean():.1f} s  "
              f"(input: {d['dur_before'].mean():.1f} s)")
        print(f"\n  If dsp_pct_rms < 70 %, consider:")
        print(f"    · Raising SILENCE_DB from {SILENCE_DB} dB to −35 or −30 dB")
        print(f"      (less aggressive silence removal)")
        print(f"    · Lowering MIN_BIRD_SEGMENT from {MIN_BIRD_SEGMENT} s to 0.3 s")
        print(f"      (retains shorter call segments)")

        # Top especies más afectadas por RMS
        if "scientific_name" in df_clean.columns:
            by_sp_rms = (
                df_clean.dropna(subset=["dsp_pct_rms", "scientific_name"])
                .groupby("scientific_name")["dsp_pct_rms"]
                .mean()
                .sort_values()
                .head(10)
            )
            if not by_sp_rms.empty:
                print(f"\n  Top 10 species most affected by RMS removal:")
                for sp, pct in by_sp_rms.items():
                    print(f"    {sp:<45} {pct:5.1f}%")

    print(f"\nTo process the full dataset set: DRY_RUN = False")

In [ ]:
# Diagnostic cell — run separately after Block 3
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

CLEAN_DIR = PROCESSED_DIR / "clean_audio"

durations = []
for fp in tqdm(list(CLEAN_DIR.glob("*.wav"))[:], desc="Measuring duration"):
    try:
        d = librosa.get_duration(path=str(fp))
        durations.append({"file": fp.name, "duration": d})
    except Exception:
        durations.append({"file": fp.name, "duration": None})

df_dur = pd.DataFrame(durations).dropna()

print(f"Total measured  : {len(df_dur):,}")
print(f"Min            : {df_dur['duration'].min():.2f} s")
print(f"Max            : {df_dur['duration'].max():.2f} s")
print(f"Mean          : {df_dur['duration'].mean():.2f} s")
print(f"Meanna        : {df_dur['duration'].median():.2f} s")
print()
print("Distribution:")
bins = [0, 0.5, 1, 2, 3, 5, 10, 20, 60]
for i in range(len(bins)-1):
    n = ((df_dur['duration'] >= bins[i]) & (df_dur['duration'] < bins[i+1])).sum()
    pct = n / len(df_dur) * 100
    print(f"  {bins[i]:4.1f} – {bins[i+1]:4.1f} s : {n:4d}  ({pct:.1f}%)")

## Block 4 — RMS Normalisation

Scales all clean WAV files to a target RMS of −20 dBFS (≈ 0.1 linear). A peak ceiling of 0.99 prevents clipping after scaling.

In [ ]:
# =============================================================
# BLOCK 4: RMS audio normalisation — full WAV files
# Input : PROCESSED_DIR / "clean_audio" (WAV limpios)
# Output  : PROCESSED_DIR / "normalized"  (WAV normalizados)
# =============================================================

import numpy as np
import librosa
import soundfile as sf
import pandas as pd
import logging
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

NORM_DIR = PROCESSED_DIR / "normalized"
NORM_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================
# Parameters
# =============================================================
#
# TARGET_RMS = 0.1
#   Nivel RMS objetivo. Equivale a -20 dBFS aproximadamente,
#   estándar en audio profesional para material de entrenamiento.
#   Evita saturación (>1.0) y señales demasiado débiles (<0.01).
#
# PEAK_CEILING = 0.99
#   Tras escalar por RMS, si alguna muestra supera este valor
#   se aplica un segundo rescalado para evitar clipping.
#   0.99 deja 0.01 de margen de seguridad.
#
# MIN_RMS = 1e-6
#   Si el RMS es prácticamente cero el archivo está vacío
#   o es silencio puro — se descarta.

SR           = 32000
TARGET_RMS   = 0.1
PEAK_CEILING = 0.99
MIN_RMS      = 1e-6
N_WORKERS    = 8

# =============================================================
# Normalisation function
# =============================================================

def normalize_audio(filepath: Path, out_dir: Path) -> dict:
    """
    Two-step normalisation:
      1. RMS scaling → brings all files to the same
         average energy level (TARGET_RMS)
      2. Peak ceiling → prevents clipping from high transients
    """
    out_path = out_dir / filepath.name  # same filename, different directory

    if out_path.exists():
        return {"file": filepath.name, "status": "skip", "rms_before": None,
                "rms_after": None, "scale": None}

    try:
        y, sr = librosa.load(str(filepath), sr=SR, mono=True)
    except Exception as e:
        return {"file": filepath.name, "status": "error",
                "rms_before": None, "rms_after": None, "scale": None}

    rms_before = float(np.sqrt(np.mean(y ** 2)))

    # Discard pure silence
    if rms_before < MIN_RMS:
        return {"file": filepath.name, "status": "discarded",
                "rms_before": rms_before, "rms_after": None, "scale": None}

    # Step 1: RMS scaling
    scale = TARGET_RMS / rms_before
    y_norm = y * scale

    # Step 2: Peak ceiling — avoid clipping
    peak = np.max(np.abs(y_norm))
    if peak > PEAK_CEILING:
        y_norm = y_norm * (PEAK_CEILING / peak)
        scale  = scale * (PEAK_CEILING / peak)

    rms_after = float(np.sqrt(np.mean(y_norm ** 2)))

    sf.write(str(out_path), y_norm.astype(np.float32), SR, subtype="PCM_16")

    return {
        "file":       filepath.name,
        "status":     "ok",
        "rms_before": round(rms_before, 6),
        "rms_after":  round(rms_after,  6),
        "scale":      round(scale,       4),
    }


# =============================================================
# Parallel processing
# =============================================================

def normalize_all(clean_dir: Path, norm_dir: Path) -> pd.DataFrame:
    wav_files = sorted(clean_dir.glob("*.wav"))
    log.info(f"Archivos WAV encontrados: {len(wav_files):,}")

    records = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(normalize_audio, fp, norm_dir): fp
            for fp in wav_files
        }
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Normalising audio",
            unit="file",
        ):
            records.append(future.result())

    return pd.DataFrame(records)


# =============================================================
# Execution
# =============================================================

log.info("=" * 60)
log.info("BLOCK 4 — Audio normalisation")
log.info("=" * 60)
log.info(f"Input       : {CLEAN_DIR}")
log.info(f"Output        : {NORM_DIR}")
log.info(f"Target RMS    : {TARGET_RMS}  (~{20*np.log10(TARGET_RMS):.0f} dBFS)")
log.info(f"Peak ceiling  : {PEAK_CEILING}")
log.info(f"Workers       : {N_WORKERS}")

df_norm = normalize_all(CLEAN_DIR, NORM_DIR)

# =============================================================
# Summary
# =============================================================

ok   = df_norm[df_norm["status"] == "ok"]
skip = df_norm[df_norm["status"] == "skip"]
desc = df_norm[df_norm["status"] == "discarded"]
err  = df_norm[df_norm["status"] == "error"]

print(f"\n{'─'*50}")
print(f"Normalised OK  : {len(ok):,}")
print(f"Already existed      : {len(skip):,}")
print(f"Discarded      : {len(desc):,}  (pure silence)")
print(f"Errors          : {len(err):,}")
print(f"\nRMS before — mean: {ok['rms_before'].mean():.4f}  "
      f"std: {ok['rms_before'].std():.4f}")
print(f"RMS after  — mean: {ok['rms_after'].mean():.4f}  "
      f"std: {ok['rms_after'].std():.4f}")
print(f"Scale factor — mean: {ok['scale'].mean():.2f}  "
      f"min: {ok['scale'].min():.2f}  max: {ok['scale'].max():.2f}")

## Block 5 — Chunking (3 s, 50 % overlap)

Slices normalised WAV files into fixed-length 5-second chunks with 50 % overlap. Chunks shorter than `MIN_SEC` are discarded; shorter final chunks are zero-padded to the full length.

In [ ]:
# =============================================================
# BLOCK 5: Chunking — 5 s windows with 50 % overlap
# Input : PROCESSED_DIR / "normalized" (WAV normalizados)
# Output  : PROCESSED_DIR / "chunks"     (WAV de 5s exactos)
# =============================================================

import numpy as np
import librosa
import soundfile as sf
import pandas as pd
import logging
from pathlib import Path
from tqdm.auto import tqdm

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

CHUNK_DIR = PROCESSED_DIR / "chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================
# Parameters
# =============================================================

SR         = 22050
CHUNK_SEC  = 3.0
OVERLAP    = 0.5
MIN_SEC    = 1.0
CHUNK_SAMP = int(CHUNK_SEC * SR)          # 110,250 samples
HOP_SAMP   = int(CHUNK_SAMP * (1 - OVERLAP))  # 55,125 samples

# =============================================================
# Chunking
# =============================================================

def _pad_or_chunk(y: np.ndarray, sr: int) -> list[np.ndarray]:
    n = len(y)

    if n < int(MIN_SEC * sr):
        return []

    if n < CHUNK_SAMP:
        padded      = np.zeros(CHUNK_SAMP, dtype=np.float32)
        padded[:n]  = y
        return [padded]

    chunks, start = [], 0
    while start < n:
        chunk = y[start: start + CHUNK_SAMP]
        if len(chunk) < CHUNK_SAMP:
            if len(chunk) >= int(MIN_SEC * sr):
                padded          = np.zeros(CHUNK_SAMP, dtype=np.float32)
                padded[:len(chunk)] = chunk
                chunks.append(padded)
            break
        chunks.append(chunk.astype(np.float32))
        start += HOP_SAMP

    return chunks


def process_chunks(norm_dir: Path, chunk_dir: Path) -> pd.DataFrame:
    wav_files    = sorted(norm_dir.glob("*.wav"))
    records      = []
    total_chunks = 0

    for fp in tqdm(wav_files, desc="Chunking audio files", unit="file"):
        try:
            y, sr = librosa.load(str(fp), sr=SR, mono=True)
        except Exception as e:
            log.debug(f"Error loading {fp.name}: {e}")
            records.append({"original": fp.name, "n_chunks": 0,
                            "status": "error"})
            continue

        chunks = _pad_or_chunk(y, sr)

        if not chunks:
            records.append({"original": fp.name, "n_chunks": 0,
                            "status": "discarded"})
            continue

        for i, chunk in enumerate(chunks):
            out_path = chunk_dir / f"{fp.stem}_chunk{i:03d}.wav"
            if not out_path.exists():
                sf.write(str(out_path), chunk, SR, subtype="PCM_16")

        records.append({"original": fp.name, "n_chunks": len(chunks),
                        "status": "ok"})
        total_chunks += len(chunks)

    df = pd.DataFrame(records)
    log.info(f"Files processed      : {(df['status']=='ok').sum():,}")
    log.info(f"Discarded (<{MIN_SEC}s) : {(df['status']=='discarded').sum():,}")
    log.info(f"Chunks generated         : {total_chunks:,}")
    return df


# =============================================================
# Execution
# =============================================================

log.info("=" * 60)
log.info("BLOCK 5 — Chunking to 5 s")
log.info("=" * 60)
log.info(f"Input    : {NORM_DIR}")
log.info(f"Output     : {CHUNK_DIR}")
log.info(f"Chunk size : {CHUNK_SEC}s  ({CHUNK_SAMP:,} samples)")
log.info(f"Overlap    : {OVERLAP*100:.0f}%  (hop = {HOP_SAMP:,} samples)")
log.info(f"Minimum     : {MIN_SEC}s")

df_chunks = process_chunks(NORM_DIR, CHUNK_DIR)

ok_df = df_chunks[df_chunks["status"] == "ok"]
print(f"\nChunks per file:")
print(f"  Min   : {ok_df['n_chunks'].min()}")
print(f"  Max   : {ok_df['n_chunks'].max()}")
print(f"  Mean : {ok_df['n_chunks'].mean():.1f}")
print(f"  Total : {ok_df['n_chunks'].sum():,}")

## Block 6 — Log-Mel Spectrograms → Normalised Tensors

Converts 5-second WAV chunks to log-Mel spectrogram tensors, computes global per-bin z-score statistics (Welford online algorithm), normalises all tensors, and builds the dataset index with an 80/10/10 stratified train/val/test split.

Output tensor shape: **(128, 431, 1)** — `(mel_bins, time_frames, channel)`, channels-last for Keras/TensorFlow.

In [ ]:
# =============================================================
# BLOCK 6: Log-Mel spectrograms → normalised tensors
# Input  : PROCESSED_DIR / "chunks"       (5 s WAV files)
# Output : PROCESSED_DIR / "spectrograms" (.npy tensors)
#          DATA_DIR / "norm_stats.parquet" (per-bin mean/std)
#          DATA_DIR / "dataset_index.parquet" (full index)
# =============================================================

import numpy as np
import librosa
import pandas as pd
import logging
import time
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

SPEC_DIR        = PROCESSED_DIR / "spectrograms"
NORM_STATS_PATH = DATA_DIR / "norm_stats.parquet"
DATASET_INDEX   = DATA_DIR / "dataset_index.parquet"

SPEC_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================
# Spectral parameters
# =============================================================
#
# N_MELS = 128, F_MIN = 300, F_MAX = 10000
#   Matches the band-pass filter in Block 3 exactly.
#   Mel bins outside the filtered band carry no signal.
#
# N_FFT = 1024, HOP = 256 @ SR = 22050 Hz
#   Temporal resolution  : 256/22050 ≈ 11.6 ms/frame
#   Frequency resolution : 22050/1024 ≈ 21.5 Hz/bin
#   5 s chunk → (110250/256) + 1 = 431 frames
#   Final tensor shape   : (128, 431, 1) — channels-last for Keras
#
# TOP_DB = 80
#   Maximum dynamic range. Values below (peak − 80 dB) are clamped.
#   Removes residual background noise without losing vocal content.
#
# Global z-score (Welford online)
#   Computed per Mel bin (frequency axis), not globally.
#   Each of the 128 bins has its own mean and std.
#   Preserves the relative spectral shape across bins while
#   equalising the absolute scale.

SR        = 22050   # must match the chunking SR in Block 5
N_MELS    = 128
F_MIN     = 300.0
F_MAX     = 10000.0
N_FFT     = 1024
HOP       = 256
TOP_DB    = 80
N_WORKERS = 8

# Expected shape: (128, 431)
T_FRAMES = int(5.0 * SR / HOP) + 1


# =============================================================
# 1. Audio → log-Mel spectrogram
# =============================================================

def audio_to_logmel(y: np.ndarray, sr: int) -> np.ndarray:
    """
    Returns a log-Mel spectrogram with shape (N_MELS, T_FRAMES).
    Pads or truncates to guarantee a fixed shape.
    """
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=N_FFT,
        hop_length=HOP,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)

    # Ensure fixed shape
    T = log_mel.shape[1]
    if T < T_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, T_FRAMES - T)),
                         mode="constant", constant_values=log_mel.min())
    else:
        log_mel = log_mel[:, :T_FRAMES]

    return log_mel.astype(np.float32)   # (128, 431)


# =============================================================
# 2. Phase 1 — generate raw spectrograms
# =============================================================

def _process_one(fp: Path, spec_dir: Path) -> dict | None:
    """Load a WAV chunk and save its spectrogram as a .npy file."""
    out_path = spec_dir / (fp.stem + ".npy")
    if out_path.exists():
        return {"file": fp.name, "spec": str(out_path), "status": "skip"}

    try:
        y, sr = librosa.load(str(fp), sr=SR, mono=True)
        spec  = audio_to_logmel(y, sr)
        np.save(str(out_path), spec)
        return {"file": fp.name, "spec": str(out_path), "status": "ok"}
    except Exception as e:
        log.debug(f"Error on {fp.name}: {e}")
        return {"file": fp.name, "spec": None, "status": "error"}


def generate_spectrograms(chunk_dir: Path, spec_dir: Path) -> pd.DataFrame:
    chunk_files = sorted(chunk_dir.glob("*.wav"))
    log.info(f"Chunks found: {len(chunk_files):,}")

    records = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(_process_one, fp, spec_dir): fp
            for fp in chunk_files
        }
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Generating spectrograms",
            unit="chunk",
        ):
            records.append(future.result())

    return pd.DataFrame(records)


# =============================================================
# 3. Phase 2 — compute global stats (Welford online)
# =============================================================

def compute_global_stats(spec_dir: Path) -> tuple[np.ndarray, np.ndarray]:
    """
    Computes per-Mel-bin mean and std over ALL spectrograms.
    Handles shape variants (128,T), (1,128,T), (128,T,1) robustly.
    Retries on PermissionError (file still locked by writer).
    """
    npy_files = sorted(spec_dir.glob("*.npy"))
    log.info(f"Computing stats over {len(npy_files):,} spectrograms...")

    n_acc    = 0
    mean_acc = np.zeros(N_MELS, dtype=np.float64)
    M2_acc   = np.zeros(N_MELS, dtype=np.float64)

    for fp in tqdm(npy_files, desc="Computing stats", unit="spec"):
        spec = None
        for attempt in range(5):
            try:
                spec = np.load(str(fp))
                break
            except PermissionError:
                time.sleep(0.2 * (attempt + 1))
            except Exception as e:
                log.debug(f"Error reading {fp.name}: {e}")
                break

        if spec is None:
            continue

        # Normalise shape to (128, T)
        if spec.ndim == 3 and spec.shape[0] == 1:
            spec = spec[0]           # (1, 128, T) → (128, T)
        elif spec.ndim == 3 and spec.shape[2] == 1:
            spec = spec[:, :, 0]     # (128, T, 1) → (128, T)

        for t in range(spec.shape[1]):
            frame     = spec[:, t].astype(np.float64)
            n_acc    += 1
            delta     = frame - mean_acc
            mean_acc += delta / n_acc
            M2_acc   += delta * (frame - mean_acc)

    global_mean = mean_acc.astype(np.float32)
    global_std  = np.sqrt(M2_acc / max(n_acc - 1, 1)).astype(np.float32)
    global_std  = np.maximum(global_std, 1e-6)

    log.info(f"Frames processed   : {n_acc:,}")
    log.info(f"Mean (across bins) : {global_mean.mean():.3f} dB")
    log.info(f"Std  (across bins) : {global_std.mean():.3f} dB")

    return global_mean, global_std


# =============================================================
# 4. Phase 3 — apply z-score and save final tensors
# =============================================================

def _normalize_and_save(fp: Path, mean: np.ndarray, std: np.ndarray) -> bool:
    """
    Loads a raw .npy, applies per-bin z-score, adds a channel
    dimension → shape (128, 431, 1). Overwrites the .npy in place.
    """
    try:
        spec = np.load(str(fp))                           # (128, T)
        norm = (spec - mean[:, None]) / std[:, None]      # z-score per bin
        norm = norm[:, :, np.newaxis]                     # (128, T, 1)
        np.save(str(fp), norm.astype(np.float32))
        return True
    except Exception as e:
        log.debug(f"Error normalising {fp.name}: {e}")
        return False


def apply_zscore(
    spec_dir:    Path,
    global_mean: np.ndarray,
    global_std:  np.ndarray,
) -> None:
    npy_files = sorted(spec_dir.glob("*.npy"))
    log.info(f"Applying z-score to {len(npy_files):,} tensors...")

    ok = 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(_normalize_and_save, fp, global_mean, global_std): fp
            for fp in npy_files
        }
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Normalising tensors",
            unit="tensor",
        ):
            if future.result():
                ok += 1
    log.info(f"Tensors normalised: {ok:,} / {len(npy_files):,}")


# =============================================================
# 5. Phase 4 — build dataset index
# =============================================================

def build_dataset_index(
    spec_dir:    Path,
    chunk_dir:   Path,
    df_filtered: pd.DataFrame,
) -> pd.DataFrame:
    """
    Builds a DataFrame with one row per tensor:
    - path to the .npy tensor
    - species (scientific_name + common_name)
    - numeric label for training
    - split assignment: train/val/test (80/10/10 stratified per species)
    """
    npy_files = sorted(spec_dir.glob("*.npy"))

    # Build species → label mapping (alphabetical order)
    species_sorted = sorted(df_filtered["scientific_name"].unique())
    label_map      = {sp: i for i, sp in enumerate(species_sorted)}

    records = []
    for fp in npy_files:
        # Extract xc_id from filename: XC123456_Species_clean_chunk001.npy
        stem    = fp.stem
        xc_id   = stem.split("_")[0].replace("XC", "")
        chunk_i = int(stem.split("chunk")[-1]) if "chunk" in stem else 0

        match = df_filtered[df_filtered["id"].astype(str) == xc_id]
        if match.empty:
            continue

        sci_name    = match.iloc[0]["scientific_name"]
        common_name = match.iloc[0].get("en", "")
        label       = label_map.get(sci_name, -1)

        records.append({
            "tensor_path"    : str(fp),
            "xc_id"          : xc_id,
            "chunk_idx"      : chunk_i,
            "scientific_name": sci_name,
            "common_name"    : common_name,
            "label"          : label,
        })

    df_idx = pd.DataFrame(records)

    # Stratified 80/10/10 split per species
    np.random.seed(42)
    splits = []
    for sp, grp in df_idx.groupby("scientific_name"):
        n      = len(grp)
        idx    = grp.index.tolist()
        np.random.shuffle(idx)
        n_val  = max(1, int(n * 0.10))
        n_test = max(1, int(n * 0.10))
        for i, row_idx in enumerate(idx):
            if i < n_val:
                splits.append((row_idx, "val"))
            elif i < n_val + n_test:
                splits.append((row_idx, "test"))
            else:
                splits.append((row_idx, "train"))

    split_df        = pd.DataFrame(splits, columns=["idx", "split"]).set_index("idx")
    df_idx["split"] = split_df["split"]

    log.info(f"Total tensors indexed : {len(df_idx):,}")
    log.info(f"Species               : {df_idx['scientific_name'].nunique():,}")
    log.info(f"Train                 : {(df_idx['split']=='train').sum():,}")
    log.info(f"Val                   : {(df_idx['split']=='val').sum():,}")
    log.info(f"Test                  : {(df_idx['split']=='test').sum():,}")

    return df_idx


# =============================================================
# 6. Execution
# =============================================================

log.info("=" * 60)
log.info("BLOCK 6 — Log-Mel spectrograms + normalised tensors")
log.info("=" * 60)
log.info(f"Input  : {CHUNK_DIR}")
log.info(f"Output : {SPEC_DIR}")
log.info(f"Shape  : ({N_MELS}, {T_FRAMES}, 1) per tensor")

CHUNK_DIR = PROCESSED_DIR / "chunks"

# Phase 1: generate raw spectrograms
df_specs = generate_spectrograms(CHUNK_DIR, SPEC_DIR)
ok_specs  = (df_specs["status"].isin(["ok", "skip"])).sum()
log.info(f"Spectrograms available: {ok_specs:,}")

# Phase 2: global stats
global_mean, global_std = compute_global_stats(SPEC_DIR)

# Save stats — required at inference time
pd.DataFrame({
    "mel_bin": np.arange(N_MELS),
    "mean":    global_mean,
    "std":     global_std,
}).to_parquet(NORM_STATS_PATH, index=False)
log.info(f"Stats saved: {NORM_STATS_PATH}")

# Phase 3: z-score + channel dimension
apply_zscore(SPEC_DIR, global_mean, global_std)

# Phase 4: dataset index
if "df_filtered" not in dir():
    df_filtered = pd.read_parquet(FILTERED_PARQUET_PATH)

df_index = build_dataset_index(SPEC_DIR, CHUNK_DIR, df_filtered)
df_index.to_parquet(DATASET_INDEX, index=False)
log.info(f"Index saved: {DATASET_INDEX}")

# =============================================================
# 7. Final verification
# =============================================================

sample = [np.load(str(p)) for p in list(SPEC_DIR.glob("*.npy"))[:200]]
means  = [s.mean() for s in sample]
stds   = [s.std()  for s in sample]

print(f"\n{'─'*50}")
print(f"Post-normalisation check (200 tensors):")
print(f"  Global mean  : {np.mean(means):.4f}  (expected ≈ 0.0)")
print(f"  Global std   : {np.mean(stds):.4f}   (expected ≈ 1.0)")
print(f"  Tensor shape : {sample[0].shape}  (expected ({N_MELS}, {T_FRAMES}, 1))")
print(f"\nFiles generated:")
print(f"  .npy tensors  : {SPEC_DIR}")
print(f"  Norm stats    : {NORM_STATS_PATH}")
print(f"  Dataset index : {DATASET_INDEX}")
print(f"\n⚠️  norm_stats.parquet is required at inference —")
print(f"   apply the same normalisation used during training.")

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
from tqdm.auto import tqdm

def convert_one(fp: Path) -> str:
    spec = np.load(str(fp))
    if spec.ndim == 3 and spec.shape[0] == 1:
        spec = np.transpose(spec, (1, 2, 0))   # (1,128,431) → (128,431,1)
        np.save(str(fp), spec.astype(np.float32))
        return "converted"
    elif spec.ndim == 3 and spec.shape[2] == 1:
        return "already_ok"
    else:
        return f"unexpected:{spec.shape}"

npy_files = sorted(SPEC_DIR.glob("*.npy"))
print(f"Tensors found: {len(npy_files):,}")

results = {"converted": 0, "already_ok": 0, "error": 0}

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(convert_one, fp): fp for fp in npy_files}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Converting"):
        try:
            r = future.result()
            key = r if r in results else "error"
            results[key] += 1
        except Exception:
            results["error"] += 1

print(f"\nConverted  : {results['converted']:,}")
print(f"Already ok : {results['already_ok']:,}")
print(f"Errors     : {results['error']:,}")

# Verify
sample = np.load(str(npy_files[0]))
print(f"\nShape: {sample.shape}  (expected (128, 431, 1))")

## Block 7 — Class Balancing by Chunk Quality
Scores every .npy tensor by mean spectral energy, then caps each species at N_TARGET chunks (default: p75 of the chunks-per-species distribution), keeping only the highest-quality ones. Writes a filtered dataset_index_balanced.parquet for the training notebook and optionally deletes the rejected tensors from disk.

Key parameters: FORCE_TARGET (int or None) · TARGET_PERCENTILE (75) · DELETE_REJECTED (True/False) · N_SCORE_WORKERS (8)
Quality metric: score = mean(tensor) — after global z-score normalisation, higher mean energy indicates more bird vocalisation and less silence/noise.
Outputs:

dataset_index.parquet — updated in-place with a boolean kept column.
dataset_index_balanced.parquet — kept=True only; this is the file the training notebook should load.

In [ ]:
# =============================================================
# BLOCK 7 — Class balancing by chunk quality
#
# Problem: species with more recordings on Xeno-Canto generate
# more chunks, creating severe imbalance (some species have
# 10× more tensors than others).
#
# Solution: for each species with excess chunks, keep only the
# N_TARGET best ones, using the mean energy of the normalized
# spectrogram as a quality proxy.
#
# Quality metric:
#   score = mean(tensor)   →  a tensor with higher relative energy
#   has more bird content and less silence/noise.
#   (Z-score already centered the data, so higher scores
#    indicate frames with greater spectral activity.)
#
# N_TARGET default = 75th percentile of the chunks-per-species
# distribution. You can force a fixed value with
# FORCE_TARGET = <int> or None to use the percentile.
#
# DELETE_REJECTED = True  →  deletes discarded .npy files from disk
# DELETE_REJECTED = False →  only marks the "kept" column in the index
# =============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import logging

log = logging.getLogger(__name__)

# =============================================================
# Parameters — adjust here
# =============================================================

FORCE_TARGET      = None  # int or None → uses automatic percentile
TARGET_PERCENTILE = 75    # percentile of chunks/species as ceiling
DELETE_REJECTED   = True  # True: deletes .npy; False: only marks index
N_SCORE_WORKERS   = 8     # threads for parallel score computation

DATASET_INDEX  = DATA_DIR / "dataset_index.parquet"
BALANCED_INDEX = DATA_DIR / "dataset_index_balanced.parquet"

# =============================================================
# 1. Load index
# =============================================================

df = pd.read_parquet(DATASET_INDEX)
log.info(f"Index loaded: {len(df):,} chunks, {df['scientific_name'].nunique():,} species")

chunks_per_species = df.groupby("scientific_name").size()
log.info(f"\nChunks-per-species distribution:")
log.info(f"  Min    : {chunks_per_species.min():,}")
log.info(f"  Median : {chunks_per_species.median():.0f}")
log.info(f"  p75    : {int(np.percentile(chunks_per_species, 75)):,}")
log.info(f"  p90    : {int(np.percentile(chunks_per_species, 90)):,}")
log.info(f"  Max    : {chunks_per_species.max():,}")

# =============================================================
# 2. Determine N_TARGET
# =============================================================

if FORCE_TARGET is not None:
    N_TARGET = int(FORCE_TARGET)
    log.info(f"N_TARGET forced: {N_TARGET}")
else:
    N_TARGET = int(np.percentile(chunks_per_species, TARGET_PERCENTILE))
    log.info(f"N_TARGET (p{TARGET_PERCENTILE}): {N_TARGET}")

species_over  = (chunks_per_species > N_TARGET).sum()
species_under = (chunks_per_species <= N_TARGET).sum()
log.info(f"Species with excess (>{N_TARGET} chunks): {species_over}")
log.info(f"Species within limit                    : {species_under}")

# =============================================================
# 3. Compute quality score for each tensor
#    score = mean of the normalized tensor
#    (higher → more spectral activity → less silence)
# =============================================================

def score_tensor(tensor_path: str) -> float:
    """
    Loads the .npy tensor and returns its mean energy as a score.
    Expected shape post-conversion: (128, 431, 1) or (1, 128, 431).
    """
    try:
        t = np.load(tensor_path)
        return float(t.mean())
    except Exception:
        return -np.inf  # if unreadable, treat as worst

log.info(f"\nComputing quality scores for {len(df):,} tensors...")

scores = {}
with ThreadPoolExecutor(max_workers=N_SCORE_WORKERS) as pool:
    futures = {
        pool.submit(score_tensor, row["tensor_path"]): idx
        for idx, row in df.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Scoring", unit="tensor"):
        idx = futures[future]
        scores[idx] = future.result()

df["quality_score"] = pd.Series(scores)
log.info(f"Scores computed. Global mean: {df['quality_score'].mean():.4f}")

# =============================================================
# 4. Selection: for each species with excess, keep the
#    N_TARGET chunks with the highest quality_score
# =============================================================

kept_indices = []

for species, grp in df.groupby("scientific_name"):
    if len(grp) <= N_TARGET:
        # Species within limit: keep all
        kept_indices.extend(grp.index.tolist())
    else:
        # Species with excess: keep the N_TARGET best
        top_idx = grp.nlargest(N_TARGET, "quality_score").index.tolist()
        kept_indices.extend(top_idx)

df["kept"] = False
df.loc[kept_indices, "kept"] = True

kept    = df["kept"].sum()
dropped = (~df["kept"]).sum()
log.info(f"\nBalancing result:")
log.info(f"  Chunks kept    : {kept:,}")
log.info(f"  Chunks dropped : {dropped:,}")
log.info(f"  Reduction      : {dropped / len(df) * 100:.1f}%")

# Verify post-balancing distribution
balanced_counts = df[df["kept"]].groupby("scientific_name").size()
log.info(f"\nPost-balancing distribution:")
log.info(f"  Min    : {balanced_counts.min():,}")
log.info(f"  Median : {balanced_counts.median():.0f}")
log.info(f"  Max    : {balanced_counts.max():,}")

# =============================================================
# 5. (Optional) Delete rejected tensors from disk
# =============================================================

if DELETE_REJECTED:
    rejected_paths = df.loc[~df["kept"], "tensor_path"].tolist()
    deleted = 0
    errors  = 0
    for p in tqdm(rejected_paths, desc="Deleting tensors", unit="file"):
        try:
            Path(p).unlink(missing_ok=True)
            deleted += 1
        except Exception as e:
            log.debug(f"Could not delete {p}: {e}")
            errors += 1
    log.info(f"Deleted: {deleted:,} | Errors: {errors:,}")
else:
    log.info("DELETE_REJECTED=False → .npy files intact on disk. Only the index reflects the balancing.")

# =============================================================
# 6. Save balanced index
#    The full index (with 'kept' column) is saved to
#    dataset_index.parquet (updated in-place).
#    The filtered index (kept=True only) is saved to
#    dataset_index_balanced.parquet → this is what the model uses.
# =============================================================

df.to_parquet(DATASET_INDEX, index=False)
df[df["kept"]].drop(columns=["kept"]).to_parquet(BALANCED_INDEX, index=False)

log.info(f"\nFull index     (with 'kept' column) → {DATASET_INDEX}")
log.info(f"Balanced index (kept=True only)      → {BALANCED_INDEX}")
log.info(f"The training notebook should load: {BALANCED_INDEX.name}")

# =============================================================
# 7. Per-species report — top 10 most trimmed
# =============================================================

report = pd.DataFrame({
    "before" : chunks_per_species,
    "after"  : balanced_counts,
}).fillna(0).astype(int)
report["trimmed"] = report["before"] - report["after"]
report = report.sort_values("trimmed", ascending=False)

print(f"\n{'─'*55}")
print(f"Top 10 most trimmed species:")
print(report.head(10).to_string())
print(f"{'─'*55}")
print(f"\nN_TARGET used    : {N_TARGET} chunks/species")
print(f"Total chunks     : {len(df):,}  →  {kept:,} (balanced)")
print(f"Balanced index   : {BALANCED_INDEX}")
